In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

train_file = '/content/data/fingerprints/android_session_1_1.csv'
test_file = '/content/data/fingerprints/android_session_2_2.csv'

# Carica i dati
train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

print("Dati caricati con successo!")
print(f"Dimensioni Training Set: {train_df.shape}")
print(f"Dimensioni Test Set: {test_df.shape}\n")

# Prepara i dati per Scikit-learn
# Separiamo le features (i segnali RSSI) dalle etichette (stanza e opera d'arte)
X_train = train_df.drop(columns=['room', 'artwork'])
y_train_room = train_df['room']
y_train_artwork = train_df['artwork']

X_test = test_df.drop(columns=['room', 'artwork'])
y_test_room = test_df['room']
y_test_artwork = test_df['artwork']

# Le etichette delle opere sono stringhe (es. "61"), ma i modelli ML preferiscono numeri.
# Usiamo un LabelEncoder per convertirle in interi (0, 1, 2, ...).
artwork_encoder = LabelEncoder()
y_train_artwork_encoded = artwork_encoder.fit_transform(y_train_artwork)
y_test_artwork_encoded = artwork_encoder.transform(y_test_artwork)

print("Dati pronti per l'addestramento.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("Avvio addestramento di base del Random Forest...")

# Crea il modello con parametri di default
# random_state=42 serve a garantire che il risultato sia sempre lo stesso
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1) # n_jobs=-1 usa tutti i processori disponibili

# Addestra il modello per predire l'OPERA D'ARTE
rf_model.fit(X_train, y_train_artwork_encoded)

# Fai le previsioni sul test set
predictions = rf_model.predict(X_test)

# Calcola l'accuratezza
accuracy = accuracy_score(y_test_artwork_encoded, predictions)

print("\n--- Risultati del Modello di Base ---")
print(f"Artwork Top-1 Accuracy (BAR 1,2): {accuracy * 100:.2f}%")

# Confronto con la baseline
print(f"Baseline 57-NN (dal paper): 85.70%")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import os

print("Avvio addestramento e salvataggio del Random Forest...")

# Addestramento e Previsione (come prima)
# Creiamo il modello per la STANZA
rf_room = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_room.fit(X_train, y_train_room)
room_predictions = rf_room.predict(X_test)
room_accuracy = accuracy_score(y_test_room, room_predictions)

# Creiamo il modello per l'OPERA D'ARTE
rf_artwork = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_artwork.fit(X_train, y_train_artwork_encoded)
artwork_predictions_encoded = rf_artwork.predict(X_test)
artwork_accuracy = accuracy_score(y_test_artwork_encoded, artwork_predictions_encoded)

# Per calcolare la Top-3 Accuracy, abbiamo bisogno delle probabilità
artwork_probabilities = rf_artwork.predict_proba(X_test)
top3_predictions = artwork_probabilities.argsort(axis=1)[:, -3:]

# Controlliamo se la risposta giusta è nelle prime 3
correct_top3 = [y_test_artwork_encoded[i] in top3_predictions[i] for i in range(len(y_test_artwork_encoded))]
top3_accuracy = sum(correct_top3) / len(correct_top3)


# Salvataggio dei Risultati

output_folder = "/content/drive/My Drive/Tesi Magistrale/Risultati_RandomForest/"
os.makedirs(output_folder, exist_ok=True) # Crea la cartella se non esiste

# Crea un DataFrame con i risultati
results_summary = {
    'Dataset': ['BAR 1,2'],
    'Algorithm': ['RandomForest_Default'],
    'Room Accuracy': [room_accuracy * 100],
    'Artwork Top-1 Accuracy': [artwork_accuracy * 100],
    'Artwork Top-3 Accuracy': [top3_accuracy * 100]
}
results_df = pd.DataFrame(results_summary)

# Salva il DataFrame in un file CSV
output_file = os.path.join(output_folder, 'summary_BAR_1_2.csv')
results_df.to_csv(output_file, index=False, float_format='%.2f')

print("\n--- Risultati Finali ---")
print(results_df.to_string(index=False))
print(f"\n Risultati salvati con successo in: {output_file}")

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import time

print("Avvio dell'ottimizzazione degli iperparametri...")
start_time = time.time()

param_grid = {
    'n_estimators': [100, 200],      # Quanti alberi usare
    'max_depth': [10, 20, None],     # Profondità massima di ogni albero (None = illimitata)
    'min_samples_split': [2, 5]      # Numero minimo di campioni per dividere un nodo
}

# Creiamo il modello base di Random Forest
# Lo useremo per l'ottimizzazione dell'opera d'arte, che è il compito più difficile
rf_for_grid = RandomForestClassifier(random_state=42, n_jobs=-1)

# Configuriamo GridSearchCV
# cv=3 significa che userà la "Cross-Validation" a 3 fold per valutare ogni combinazione
# verbose=2 mostra i progressi durante l'esecuzione
grid_search = GridSearchCV(estimator=rf_for_grid, param_grid=param_grid, cv=3, verbose=2, scoring='accuracy')

# Avviamo la ricerca! Questo potrebbe richiedere alcuni minuti.
grid_search.fit(X_train, y_train_artwork_encoded)

end_time = time.time()
print(f"\nRicerca completata in {((end_time - start_time) / 60):.2f} minuti.")

# Mostriamo i risultati
print("\n--- Risultati dell'Ottimizzazione ---")
print(f"Migliori parametri trovati: {grid_search.best_params_}")
print(f"Miglior punteggio di accuratezza (in cross-validation): {grid_search.best_score_ * 100:.2f}%")

# Valutiamo il modello OTTIMIZZATO sul test set
best_rf_model = grid_search.best_estimator_
predictions = best_rf_model.predict(X_test)
optimized_accuracy = accuracy_score(y_test_artwork_encoded, predictions)

print("\n--- Confronto Performance su Test Set (BAR 1,2) ---")
print(f"Accuratezza Modello di Default: {accuracy * 100:.2f}%") # 'accuracy' calcolato nel blocco precedente
print(f"Accuratezza Modello Ottimizzato: {optimized_accuracy * 100:.2f}%")

In [ ]:
import pandas as pd
import os
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

print("Avvio della suite di esperimenti completa con il Random Forest ottimizzato...")

data_dir = '/content/data/fingerprints/'
files = [
    'android_session_1_1.csv',
    'android_session_2_2.csv',
    'ios_session_1_3.csv',
    'ios_session_2_4.csv'
]

# Creiamo tutte le 12 combinazioni di train/test
experiments = []
for train_file in files:
    for test_file in files:
        if train_file != test_file:
            experiments.append((train_file, test_file))

# Lista per salvare tutti i risultati
all_results = []

# Definisci i parametri ottimali che abbiamo trovato
best_params = {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}

# Ciclo su tutti gli esperimenti
for i, (train_file, test_file) in enumerate(experiments):
    print(f"\n--- Esecuzione Esperimento {i+1}/{len(experiments)}: Train={train_file}, Test={test_file} ---")
    start_time = time.time()

    # Carica i dati
    train_df = pd.read_csv(os.path.join(data_dir, train_file))
    test_df = pd.read_csv(os.path.join(data_dir, test_file))

    # Prepara i dati
    X_train = train_df.drop(columns=['room', 'artwork'])
    y_train_room = train_df['room']
    y_train_artwork = train_df['artwork']
    X_test = test_df.drop(columns=['room', 'artwork'])
    y_test_room = test_df['room']
    y_test_artwork = test_df['artwork']

    artwork_encoder = LabelEncoder()
    y_train_artwork_encoded = artwork_encoder.fit_transform(y_train_artwork)
    y_test_artwork_encoded = artwork_encoder.transform(y_test_artwork)

    # Addestra e valuta il modello per la STANZA
    rf_room = RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)
    rf_room.fit(X_train, y_train_room)
    room_accuracy = accuracy_score(y_test_room, rf_room.predict(X_test))

    # Addestra e valuta il modello per l'OPERA D'ARTE
    rf_artwork = RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)
    rf_artwork.fit(X_train, y_train_artwork_encoded)
    artwork_predictions = rf_artwork.predict(X_test)
    artwork_top1_accuracy = accuracy_score(y_test_artwork_encoded, artwork_predictions)

    artwork_probabilities = rf_artwork.predict_proba(X_test)
    top3_predictions = artwork_probabilities.argsort(axis=1)[:, -3:]
    correct_top3 = [y_test_artwork_encoded[j] in top3_predictions[j] for j in range(len(y_test_artwork_encoded))]
    artwork_top3_accuracy = sum(correct_top3) / len(correct_top3)

    # Salva i risultati di questo esperimento
    dataset_name = f"BAR {train_file.split('_')[-1].split('.')[0]},{test_file.split('_')[-1].split('.')[0]}"
    all_results.append({
        'Dataset': dataset_name,
        'Room Accuracy': room_accuracy * 100,
        'Artwork Top-1 Accuracy': artwork_top1_accuracy * 100,
        'Artwork Top-3 Accuracy': artwork_top3_accuracy * 100
    })

    end_time = time.time()
    print(f"Completato in {end_time - start_time:.2f} secondi.")

# Converti i risultati in un DataFrame e salvali
results_df = pd.DataFrame(all_results)
output_file = "/content/drive/My Drive/Tesi Magistrale/Risultati_RandomForest/RF_optimized_all_experiments.csv"
results_df.to_csv(output_file, index=False, float_format='%.2f')

print("\n\n--- TABELLA RIASSUNTIVA RANDOM FOREST OTTIMIZZATO ---")
print(results_df.to_string(index=False))
print(f"\n Suite di esperimenti completata! Risultati salvati in: {output_file}")